# 07 · PyIceberg: reading tables from plain Python

**Time:** ~25 min · **You will learn:** use PyIceberg to explore and query tables without a Spark cluster.

PyIceberg is a pure Python implementation of the Iceberg spec. It is ideal for notebooks, scripts, services and
small/medium data where starting Spark would be overkill.

In [ ]:
from helpers import get_spark, reset_orders

# We use Spark just once, to create a sample table the way you would in a real pipeline.
spark = get_spark("07-setup")
reset_orders(spark, "orders", n=12, commits=3)
spark.stop()

## 1. Connect to the catalog

In [ ]:
from pyiceberg.catalog import load_catalog

catalog = load_catalog("default")
print(catalog.list_namespaces())
print(catalog.list_tables("course"))

## 2. Load a table and inspect it

In [ ]:
tbl = catalog.load_table("course.orders")
print(tbl.schema())
print("partition spec:", tbl.spec())
print("current snapshot:", tbl.current_snapshot().snapshot_id)
print("snapshots:", [s.snapshot_id for s in tbl.snapshots()])

## 3. Scan the data
A `scan` is lazy; `.to_arrow()`, `.to_pandas()` or `.to_duckdb()` materialise it.

In [ ]:
df = tbl.scan().to_pandas()
print(df.shape)
df.head()

### Filters and column selection are pushed down
PyIceberg uses file statistics to skip files and only reads the selected columns.

In [ ]:
big = tbl.scan(
    row_filter="quantity >= 3 AND unit_price > 20",
    selected_fields=("order_id", "customer", "quantity", "unit_price"),
).to_pandas()
big

You can also build filters from expression objects:

In [ ]:
from pyiceberg.expressions import And, EqualTo, GreaterThanOrEqual

alice = tbl.scan(row_filter=And(EqualTo("customer", "alice"), GreaterThanOrEqual("quantity", 1))).to_arrow()
print(alice.num_rows, "rows for alice")

## 4. Time travel
Pass a `snapshot_id` to read an older version, just like `VERSION AS OF` in Spark.

In [ ]:
first = tbl.snapshots()[0].snapshot_id
print("rows then:", tbl.scan(snapshot_id=first).to_arrow().num_rows, "| rows now:", tbl.scan().to_arrow().num_rows)

## 5. SQL with DuckDB
`to_duckdb` registers the scan as a DuckDB table so you can use full SQL on it.

In [ ]:
con = tbl.scan().to_duckdb(table_name="orders")
con.execute("""
    SELECT customer, SUM(quantity * unit_price) AS revenue
    FROM orders GROUP BY customer ORDER BY revenue DESC
""").fetchdf()

## 6. Metadata, the Python way
The same metadata tables we used in Spark are available under `tbl.inspect`.

In [ ]:
tbl.inspect.snapshots().to_pandas()[["snapshot_id", "parent_id", "operation"]]

In [ ]:
tbl.inspect.files().select(["file_path", "record_count", "file_size_in_bytes"]).to_pandas()

### Your turn

Using only PyIceberg + pandas, find the **customer with the highest total revenue**
(`quantity * unit_price`) and the number of orders they placed. Assert it matches a DuckDB query on the same table.

In [ ]:
# Write your solution here


<details>
<summary>Show solution</summary>

```python
df = tbl.scan().to_pandas()
df["revenue"] = df["quantity"] * df["unit_price"]
by_customer = df.groupby("customer").agg(revenue=("revenue", "sum"), orders=("order_id", "count"))
top = by_customer.sort_values("revenue", ascending=False).iloc[0]
print(top.name, top.revenue, int(top.orders))

con = tbl.scan().to_duckdb(table_name="orders_check")
expected = con.execute(
    "SELECT customer FROM orders_check GROUP BY customer ORDER BY SUM(quantity * unit_price) DESC LIMIT 1"
).fetchone()[0]
assert top.name == expected
```

</details>

### Recap
* `load_catalog` → `load_table` → `scan(...).to_arrow() / to_pandas() / to_duckdb()`.
* Filters and column selection are pushed down; `snapshot_id` gives time travel.
* `table.inspect` exposes snapshots, files, manifests and partitions.

Next: **08 · PyIceberg: writing tables**.